In [ ]:
!pip install -q trl datasets huggingface_hub
!pip install -U bitsandbytes>=0.46.1
!pip install flash-attn --no-build-isolation

In [ ]:
import os

# Set your HF token (env var or paste directly)
HF_TOKEN = os.environ["HF_TOKEN"]
!huggingface-cli login --token $HF_TOKEN

In [ ]:
import os
import json
import random
import torch

from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# ============================================================
# Config
# ============================================================

MODEL_ID = "Lyte/tiny-aya-darija-v4"  # base model (v4 = current best)
REFINED_REPO = "Lyte/darija-translation-data-refined"
ORIGINAL_REPO = "Lyte/darija-translation-data"
OUTPUT_HF_REPO = "Lyte/tiny-aya-darija-v5"

SEED = 42
VAL_RATIO = 0.05
BIDIRECTIONAL_FRACTION = 0.20  # 20% reverse (Darija→English) pairs

random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# ============================================================
# System prompts for translation SFT
# ============================================================

SYS_EN_DR = (
    "Translate the following English text into Moroccan Darija (الدارجة المغربية). "
    "Preserve the meaning exactly. Preserve names, numbers, quoted strings, placeholders "
    "like [x], markdown, bullets, and exact phrases that must remain unchanged. "
    "Do not answer the text. Output ONLY the Darija translation."
)

SYS_DR_EN = (
    "Translate the following Moroccan Darija (الدارجة المغربية) text into English. "
    "Preserve the meaning exactly. Preserve names, numbers, quoted strings, placeholders "
    "like [x], markdown, bullets, and exact phrases that must remain unchanged. "
    "Do not answer the text. Output ONLY the English translation."
)

In [ ]:
# ============================================================
# Load & merge refined + original data
# ============================================================

# Load refined dataset (27K records, ~53% actually modified)
print("Loading refined dataset...")
refined_ds = load_dataset(REFINED_REPO, data_files="alpaca_darija_refined.jsonl",
                          split="train", token=HF_TOKEN)
print(f"  Refined: {len(refined_ds):,} records")

# Load original dataset (33K records)
print("Loading original dataset...")
original_ds = load_dataset(ORIGINAL_REPO, data_files="alpaca_darija.jsonl",
                           split="train", token=HF_TOKEN)
print(f"  Original: {len(original_ds):,} records")

# Merge: use refined for first 27K, original for remaining 6K
refined_count = len(refined_ds)
original_count = len(original_ds)

merged_records = []

# Take all refined records
for i in range(refined_count):
    row = refined_ds[i]
    merged_records.append({
        "messages_en": row["messages_en"],
        "messages": row["messages"],
        "source": row.get("source", "alpaca"),
        "refined": row.get("refined", False),
    })

# Fill remaining from original (records that weren't refined)
for i in range(refined_count, original_count):
    row = original_ds[i]
    merged_records.append({
        "messages_en": row["messages_en"],
        "messages": row["messages"],
        "source": row.get("source", "alpaca"),
        "refined": False,
    })

print(f"\nMerged dataset: {len(merged_records):,} records")
print(f"  From refined: {refined_count:,}")
print(f"  From original (unrefined): {original_count - refined_count:,}")
refined_actual = sum(1 for r in merged_records if r.get("refined"))
print(f"  Actually modified by refinement: {refined_actual:,} ({100*refined_actual/len(merged_records):.1f}%)")

In [ ]:
# ============================================================
# Build SFT training rows from merged data
# ============================================================

def build_multiturn_translation_row(en_msgs, da_msgs, sys_prompt, reverse=False):
    """Build a multi-turn translation row.
    Each EN message becomes a user turn, each DA translation an assistant turn.
    If reverse=True, swap: DA messages are user turns, EN are assistant turns.
    """
    src_msgs = da_msgs if reverse else en_msgs
    tgt_msgs = en_msgs if reverse else da_msgs

    # Pair messages into (user, assistant) turns
    pairs = []
    for src_m, tgt_m in zip(src_msgs, tgt_msgs):
        src_text = src_m["content"].strip()
        tgt_text = tgt_m["content"].strip()
        if src_text and tgt_text:
            pairs.append((src_text, tgt_text))

    if len(pairs) < 1:
        return None

    # Prompt = system + all turns except the last assistant response
    prompt = [{"role": "system", "content": sys_prompt}]
    for i, (src, tgt) in enumerate(pairs):
        prompt.append({"role": "user", "content": src})
        if i < len(pairs) - 1:
            prompt.append({"role": "assistant", "content": tgt})

    # Completion = last assistant response
    completion = [{"role": "assistant", "content": pairs[-1][1]}]

    return {"prompt": prompt, "completion": completion}

# Shuffle merged records
random.shuffle(merged_records)

# Split: val set
val_size = int(len(merged_records) * VAL_RATIO)
val_records = merged_records[:val_size]
train_records = merged_records[val_size:]

print(f"Train: {len(train_records):,} | Val: {val_size:,}")

# Build training rows — 1 multi-turn row per conversation, translation only
train_rows = []
val_rows = []

for rec in train_records:
    en_msgs = rec["messages_en"]
    da_msgs = rec["messages"]

    # Filter: minimum 2 messages (user + assistant)
    if len(en_msgs) < 2 or len(da_msgs) < 2:
        continue

    # Forward: EN → Darija (multi-turn)
    row = build_multiturn_translation_row(en_msgs, da_msgs, SYS_EN_DR)
    if row:
        train_rows.append(row)

    # 20% reverse: Darija → EN (multi-turn)
    if random.random() < BIDIRECTIONAL_FRACTION:
        rev_row = build_multiturn_translation_row(en_msgs, da_msgs, SYS_DR_EN, reverse=True)
        if rev_row:
            train_rows.append(rev_row)

# Validation: forward translation only
for rec in val_records:
    en_msgs = rec["messages_en"]
    da_msgs = rec["messages"]
    if len(en_msgs) < 2 or len(da_msgs) < 2:
        continue
    row = build_multiturn_translation_row(en_msgs, da_msgs, SYS_EN_DR)
    if row:
        val_rows.append(row)

random.shuffle(train_rows)

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)

print(f"\nFinal dataset sizes:")
print(f"  Train rows: {len(train_ds):,}")
print(f"  Val rows:   {len(val_ds):,}")
print(f"  (~1 forward + {BIDIRECTIONAL_FRACTION*100:.0f}% reverse per conversation)")

In [ ]:
# ============================================================
# Inspect a few training examples
# ============================================================

for i in range(3):
    row = train_ds[i]
    print(f"\n{'='*60}")
    print(f"Example {i}")
    print(f"{'='*60}")
    for m in row["prompt"]:
        role = m["role"]
        text = m["content"][:120]
        print(f"  [{role}]: {text}...")
    for m in row["completion"]:
        role = m["role"]
        text = m["content"][:120]
        print(f"  [{role}]: {text}...")

In [ ]:
# ============================================================
# Load tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer: {tokenizer.name_or_path}")
print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"Pad token: {tokenizer.pad_token}")

In [ ]:
# ============================================================
# QLoRA 4-bit + LoRA config (A100 optimized)
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto",
    attn_implementation="sdpa",
)

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

print(f"Model loaded: {MODEL_ID}")
print(f"LoRA r={peft_config.r}, alpha={peft_config.lora_alpha}")
print(f"Attention: flash_attention_2")

In [ ]:
# ============================================================
# Training config (H200 140GB — 256K vocab eats VRAM on logits)
# ============================================================

training_args = SFTConfig(
    output_dir="tiny-aya-darija-v5",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    weight_decay=0.05,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    eval_steps=250,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=0.3,
    report_to="none",
    seed=SEED,
    max_length=2048,
    packing=False,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)
# ============================================================
# Trainer
# ============================================================

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print(f"\n{'='*70}")
print("Darija Translation SFT (v5) — H200")
print(f"{'='*70}")
print(f"Base model:       {MODEL_ID}")
print(f"Merged data:      {len(merged_records):,} conversations")
print(f"Train rows:       {len(train_ds):,}")
print(f"Val rows:         {len(val_ds):,}")
print(f"Epochs:           {training_args.num_train_epochs}")
print(f"LoRA r:           {peft_config.r}")
print(f"Max length:       {training_args.max_length}")
print(f"LR:               {training_args.learning_rate}")
print(f"Eff batch:        {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps} = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Bidirectional:    {BIDIRECTIONAL_FRACTION*100:.0f}%")
print(f"{'='*70}\n")

In [ ]:
trainer.train()

In [ ]:
# ============================================================
# Merge LoRA + Push to HF
# ============================================================

from peft import PeftModel

# Find best checkpoint
import glob
checkpoints = sorted(glob.glob("tiny-aya-darija-v5/checkpoint-*"))
best_ckpt = checkpoints[-1] if checkpoints else "tiny-aya-darija-v5"
print(f"Using checkpoint: {best_ckpt}")

# Load base model (full precision for merge)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# Load LoRA adapter
model_with_lora = PeftModel.from_pretrained(base_model, best_ckpt)

# Merge
print("Merging LoRA adapter...")
merged_model = model_with_lora.merge_and_unload()

# Save locally
print("Saving locally...")
merged_model.save_pretrained("darija-v5-merged", safe_serialization=True)
tokenizer.save_pretrained("darija-v5-merged")

# Push to HF
print(f"Pushing to {OUTPUT_HF_REPO}...")
merged_model.push_to_hub(OUTPUT_HF_REPO, private=True)
tokenizer.push_to_hub(OUTPUT_HF_REPO, private=True)

print(f"\n✅ Pushed to https://huggingface.co/{OUTPUT_HF_REPO}")

In [ ]:
# ============================================================
# Quick test: translate a few phrases
# ============================================================

from transformers import pipeline

test_pipe = pipeline(
    "text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# Kill the max_length=20 default from generation_config so it doesn't clash with max_new_tokens
test_pipe.model.generation_config.max_length = None

GEN_PARAMS = dict(
    max_new_tokens=256,
    temperature=0.3,
    top_k=300,
    top_p=0.98,
    repetition_penalty=1.15,
    do_sample=True,
)

test_phrases = [
    "Hello, how are you?",
    "What is the capital of Morocco?",
    "Give me three tips for learning a new language.",
    "I forgot my keys at home.",
    "Explain what machine learning is in simple terms.",
]

for phrase in test_phrases:
    messages = [
        {"role": "system", "content": SYS_EN_DR},
        {"role": "user", "content": phrase},
    ]
    result = test_pipe(messages, **GEN_PARAMS)
    output = result[0]["generated_text"][-1]["content"]
    print(f"\nEN: {phrase}")
    print(f"DA: {output}")

In [ ]:
# ============================================================
# Test on Nemotron data — all splits, 10 each, save to file
# ============================================================

from datasets import load_dataset as load_ds_fresh

splits = ["math", "code", "stem"]
all_results = []

for split_name in splits:
    print(f"\n{'#'*70}")
    print(f"# SPLIT: {split_name.upper()}")
    print(f"{'#'*70}")

    ds = load_ds_fresh(
        "nvidia/Nemotron-Research-GooseReason-0.7M",
        split=split_name,
        streaming=True,
    )

    random.seed(99)
    buffer = []
    for idx, row in enumerate(ds):
        buffer.append(row)
        if idx >= 499:
            break

    samples = random.sample(buffer, 10)

    for i, sample in enumerate(samples):
        question = sample.get("question", "")
        options = sample.get("options", [])
        answer = sample.get("answer", "")

        if not question:
            print(f"\n[{split_name} Sample {i+1}] — skipped (no question)")
            continue

        input_text = question
        if options:
            for j, opt in enumerate(options):
                input_text += f"\n{chr(65+j)}. {opt}"

        input_text = input_text[:500]

        chat_msgs = [
            {"role": "system", "content": SYS_EN_DR},
            {"role": "user", "content": input_text},
        ]
        result = test_pipe(chat_msgs, **GEN_PARAMS)
        model_output = result[0]["generated_text"][-1]["content"]

        record = {
            "split": split_name,
            "sample": i + 1,
            "answer": answer,
            "en": input_text,
            "darija": model_output,
        }
        all_results.append(record)

        print(f"\n{'='*60}")
        print(f"{split_name.upper()} Sample {i+1} (answer: {answer})")
        print(f"{'='*60}")
        print(f"EN:\n{input_text}\n")
        print(f"DARIJA:\n{model_output}\n")

# Save all results to JSONL
out_path = "nemotron_darija_test_v5.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for rec in all_results:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"\n{'='*70}")
print(f"Saved {len(all_results)} results to {out_path}")
print(f"  math: {sum(1 for r in all_results if r['split']=='math')}")
print(f"  code: {sum(1 for r in all_results if r['split']=='code')}")
print(f"  stem: {sum(1 for r in all_results if r['split']=='stem')}")
print(f"{'='*70}")